<a href="https://colab.research.google.com/github/kunalchauhann/DSVLAB/blob/main/BCSE1593_Data_Cleaning_and_Preparation_Self_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BCSE1593 – Data Science and Visualization Lab

# Practical: Data Cleaning and Preparation

## Why do we need data cleaning?

A dataset may look organised in a spreadsheet, but that does not guarantee that it is ready for analysis. In a real project, data can contain blank values, special codes such as `NA` or `-999`, duplicate records, inconsistent labels, incorrect data types, and unusually large or small values.

Imagine that an online store wants to prepare its order records for a monthly sales report. If `"North"`, `"north"` and `" North "` are treated as different regions, the report may count the same region separately. If a price of `-999` is treated as a real price, the sales total will be wrong. If the same order line is counted twice, revenue may be overstated.

Our job is to identify these issues, decide how to handle them, and verify the results.

## What will we do in this practical?

We will work through a realistic order dataset and prepare it for analysis.

1. Load and inspect the raw data.
2. Identify missing values, including non-standard missing-value codes.
3. Drop or fill missing values using suitable methods.
4. Detect and remove duplicate records.
5. standardise text and column names.
6. Convert columns to suitable data types.
7. Create useful columns from existing values.
8. Check possible outliers before deciding what to do with them.
9. Create categories and indicator columns.
10. Sample and shuffle records.
11. Extract useful information from text.
12. Build a repeatable cleaning pipeline and save the result.

The coding cells are left for you to complete during the live demonstration. Read the explanation first, try the task, run the cell, and inspect the output before moving on.

# Part A – Create and Understand the Raw Data

## 1. Import the libraries

### Why are we doing this?

We need tools for working with tables, calculating statistics, representing missing numeric values, and finding patterns in text.

### Functions and modules

- `pandas` (`pd`) provides DataFrames and data-cleaning operations.
- `numpy` (`np`) provides numerical functions and values such as `np.nan`.
- `re` provides regular-expression tools for finding and replacing text patterns.

We will use the conventional aliases `pd` and `np` so that the code is shorter and familiar to other Python users.

### Live coding

Import pandas as `pd`, NumPy as `np`, and the regular-expression module as `re`.

In [1]:
# Import pandas, NumPy, and the regular-expression module
import pandas as pd
import numpy as np
import re

## 2. Create the order records

### Why are we using this dataset?

Each row represents an order line from an online store. It includes an order ID, date, customer ID, product, quantity, price, status, email and region.

The records include common issues that analysts encounter:

- Extra spaces and inconsistent capitalisation
- Missing-value placeholders such as `"."` and `"NA"`
- A special numeric code such as `"-999"`
- Repeated order records
- Numeric values stored as text

We use a built-in sample so everyone can run the notebook without configuring an external data source. The aim is to focus on cleaning methods rather than spending lab time troubleshooting downloads.

### Live coding

Run the next cell to define the sample records. This cell is supplied because it creates the raw input for the practical.

In [2]:
raw_records = [
    {"Order_ID": "ORD-1001", "Order_Date": "2026-01-02", "Customer_ID": "C101", "Product": "  wireless mouse ", "Quantity": "2", "Unit_Price": "15.75", "Status": "shipped", "Email": "riya.sharma@gmail.com", "Region": "North"},
    {"Order_ID": "ORD-1002", "Order_Date": "2026-01-02", "Customer_ID": "C102", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Pending", "Email": "AMAN@EXAMPLE.COM", "Region": "north"},
    {"Order_ID": "ORD-1003", "Order_Date": "2026-01-03", "Customer_ID": "C103", "Product": "Keyboard", "Quantity": "2", "Unit_Price": "45.00", "Status": "SHIPPED ", "Email": "neha@gmail.com", "Region": "South"},
    {"Order_ID": "ORD-1004", "Order_Date": "2026-01-03", "Customer_ID": "C104", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "delivered", "Email": " . ", "Region": "West"},
    {"Order_ID": "ORD-1005", "Order_Date": "2026-01-04", "Customer_ID": "C105", "Product": "USB-C Hub", "Quantity": "3", "Unit_Price": "29.99", "Status": "Pending", "Email": "kabir@shop.in", "Region": "East"},
    {"Order_ID": "ORD-1006", "Order_Date": "2026-01-04", "Customer_ID": "C106", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Shipped", "Email": "NA", "Region": "South"},
    {"Order_ID": "ORD-1007", "Order_Date": "2026-01-05", "Customer_ID": "C107", "Product": "Wireless Mouse", "Quantity": "5", "Unit_Price": "15.75", "Status": "delivered", "Email": "dev.kumar@gmail.com", "Region": "North"},
    {"Order_ID": "ORD-1008", "Order_Date": "2026-01-05", "Customer_ID": "C108", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "priya@example.com", "Region": "East"},
    {"Order_ID": "ORD-1009", "Order_Date": "2026-01-06", "Customer_ID": "C109", "Product": "Keyboard", "Quantity": "1", "Unit_Price": "45.00", "Status": "Shipped", "Email": "  ", "Region": "West"},
    {"Order_ID": "ORD-1010", "Order_Date": "2026-01-06", "Customer_ID": "C110", "Product": "USB-C Hub", "Quantity": "2", "Unit_Price": "29.99", "Status": "Pending", "Email": "meera@shop.in", "Region": "South"},
    {"Order_ID": "ORD-1011", "Order_Date": "2026-01-07", "Customer_ID": "C111", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "-999", "Status": "Shipped", "Email": "arjun@gmail.com", "Region": "North"},
    {"Order_ID": "ORD-1012", "Order_Date": "2026-01-07", "Customer_ID": "C112", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "sana@example.com", "Region": "East"},
    {"Order_ID": "ORD-1013", "Order_Date": "2026-01-08", "Customer_ID": "C113", "Product": "Keyboard", "Quantity": "1", "Unit_Price": "45.00", "Status": "delivered", "Email": " . ", "Region": "South"},
    {"Order_ID": "ORD-1014", "Order_Date": "2026-01-08", "Customer_ID": "C114", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "isha@example.com", "Region": "West"},
    {"Order_ID": "ORD-1015", "Order_Date": "2026-01-09", "Customer_ID": "C115", "Product": "USB-C Hub", "Quantity": "4", "Unit_Price": "29.99", "Status": "Shipped", "Email": "ROHAN@SHOP.IN", "Region": "North"},
    {"Order_ID": "ORD-1016", "Order_Date": "2026-01-09", "Customer_ID": "C116", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Pending", "Email": "tanvi@gmail.com", "Region": "East"},
    {"Order_ID": "ORD-1017", "Order_Date": "2026-01-10", "Customer_ID": "C117", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "delivered", "Email": "vishal@example.com", "Region": "South"},
    {"Order_ID": "ORD-1018", "Order_Date": "2026-01-10", "Customer_ID": "C118", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Shipped", "Email": "  ", "Region": "West"},
    {"Order_ID": "ORD-1019", "Order_Date": "2026-01-11", "Customer_ID": "C119", "Product": "Keyboard", "Quantity": "2", "Unit_Price": "45.00", "Status": "Pending", "Email": "amit@gmail.com", "Region": "North"},
    {"Order_ID": "ORD-1020", "Order_Date": "2026-01-11", "Customer_ID": "C120", "Product": "Desk Lamp", "Quantity": "1", "Unit_Price": "22.50", "Status": "Shipped", "Email": "simran@example.com", "Region": "East"},
    {"Order_ID": "ORD-1003", "Order_Date": "2026-01-03", "Customer_ID": "C103", "Product": "Keyboard", "Quantity": "2", "Unit_Price": "45.00", "Status": "SHIPPED ", "Email": "neha@gmail.com", "Region": "South"},
    {"Order_ID": "ORD-1021", "Order_Date": "2026-01-12", "Customer_ID": "C121", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Pending", "Email": "jay@example.com", "Region": "North"},
    {"Order_ID": "ORD-1022", "Order_Date": "2026-01-12", "Customer_ID": "C122", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Delivered", "Email": "kiran@gmail.com", "Region": "West"},
    {"Order_ID": "ORD-1023", "Order_Date": "2026-01-13", "Customer_ID": "C123", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "farah@example.com", "Region": "East"},
    {"Order_ID": "ORD-1024", "Order_Date": "2026-01-13", "Customer_ID": "C124", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "Shipped", "Email": "om@gmail.com", "Region": "South"},
    {"Order_ID": "ORD-1025", "Order_Date": "2026-01-14", "Customer_ID": "C125", "Product": "Keyboard", "Quantity": "1", "Unit_Price": "45.00", "Status": "Pending", "Email": "NA", "Region": "North"},
    {"Order_ID": "ORD-1026", "Order_Date": "2026-01-14", "Customer_ID": "C126", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Delivered", "Email": "siddhi@example.com", "Region": "West"},
    {"Order_ID": "ORD-1027", "Order_Date": "2026-01-15", "Customer_ID": "C127", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "rohit@example.com", "Region": "East"},
    {"Order_ID": "ORD-1028", "Order_Date": "2026-01-15", "Customer_ID": "C128", "Product": "USB-C Hub", "Quantity": "3", "Unit_Price": "29.99", "Status": "Shipped", "Email": " . ", "Region": "South"},
    {"Order_ID": "ORD-1029", "Order_Date": "2026-01-16", "Customer_ID": "C129", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Pending", "Email": "mohit@gmail.com", "Region": "North"},
    {"Order_ID": "ORD-1030", "Order_Date": "2026-01-16", "Customer_ID": "C130", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "Delivered", "Email": "anaya@example.com", "Region": "East"},
    {"Order_ID": "ORD-1031", "Order_Date": "2026-01-17", "Customer_ID": "C131", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Shipped", "Email": "  ", "Region": "West"},
    {"Order_ID": "ORD-1032", "Order_Date": "2026-01-17", "Customer_ID": "C132", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "mira@gmail.com", "Region": "South"},
    {"Order_ID": "ORD-1033", "Order_Date": "2026-01-18", "Customer_ID": "C133", "Product": "USB-C Hub", "Quantity": "2", "Unit_Price": "29.99", "Status": "Shipped", "Email": "yash@example.com", "Region": "North"},
    {"Order_ID": "ORD-1034", "Order_Date": "2026-01-18", "Customer_ID": "C134", "Product": "Keyboard", "Quantity": "1", "Unit_Price": "45.00", "Status": "Pending", "Email": "sim@example.com", "Region": "East"},
    {"Order_ID": "ORD-1035", "Order_Date": "2026-01-19", "Customer_ID": "C135", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "Shipped", "Email": "rohit@example.com", "Region": "South"},
    {"Order_ID": "ORD-1036", "Order_Date": "2026-01-19", "Customer_ID": "C136", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "NA", "Region": "East"},
    {"Order_ID": "ORD-1037", "Order_Date": "2026-01-20", "Customer_ID": "C137", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Shipped", "Email": "sahil@gmail.com", "Region": "West"},
    {"Order_ID": "ORD-1038", "Order_Date": "2026-01-20", "Customer_ID": "C138", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Pending", "Email": "isha@example.com", "Region": "North"},
    {"Order_ID": "ORD-1039", "Order_Date": "2026-01-21", "Customer_ID": "C139", "Product": "USB-C Hub", "Quantity": "2", "Unit_Price": "29.99", "Status": "Delivered", "Email": "raj@gmail.com", "Region": "South"},
    {"Order_ID": "ORD-1040", "Order_Date": "2026-01-21", "Customer_ID": "C140", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "Pending", "Email": "  ", "Region": "East"},
    {"Order_ID": "ORD-1041", "Order_Date": "2026-01-22", "Customer_ID": "C141", "Product": "Keyboard", "Quantity": "1", "Unit_Price": "45.00", "Status": "Shipped", "Email": "raj@example.com", "Region": "North"},
    {"Order_ID": "ORD-1042", "Order_Date": "2026-01-22", "Customer_ID": "C142", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Delivered", "Email": "gaurav@gmail.com", "Region": "West"},
    {"Order_ID": "ORD-1043", "Order_Date": "2026-01-23", "Customer_ID": "C143", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "zoya@example.com", "Region": "East"},
    {"Order_ID": "ORD-1044", "Order_Date": "2026-01-23", "Customer_ID": "C144", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "Shipped", "Email": "vansh@gmail.com", "Region": "South"},
    {"Order_ID": "ORD-1045", "Order_Date": "2026-01-24", "Customer_ID": "C145", "Product": "Laptop X", "Quantity": "1", "Unit_Price": "999.50", "Status": "Pending", "Email": "NA", "Region": "North"},
    {"Order_ID": "ORD-1046", "Order_Date": "2026-01-24", "Customer_ID": "C146", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Shipped", "Email": "riya@example.com", "Region": "West"},
    {"Order_ID": "ORD-1047", "Order_Date": "2026-01-25", "Customer_ID": "C147", "Product": "Desk Lamp", "Quantity": "2", "Unit_Price": "22.50", "Status": "Pending", "Email": "kavya@gmail.com", "Region": "East"},
    {"Order_ID": "ORD-1048", "Order_Date": "2026-01-25", "Customer_ID": "C148", "Product": "Wireless Mouse", "Quantity": "2", "Unit_Price": "15.75", "Status": "Shipped", "Email": "mohit@example.com", "Region": "South"},
    {"Order_ID": "ORD-1049", "Order_Date": "2026-01-26", "Customer_ID": "C149", "Product": "Keyboard", "Quantity": "1", "Unit_Price": "45.00", "Status": "Pending", "Email": "deepa@gmail.com", "Region": "North"},
    {"Order_ID": "ORD-1050", "Order_Date": "2026-01-26", "Customer_ID": "C150", "Product": "Monitor Y", "Quantity": "1", "Unit_Price": "189.99", "Status": "Delivered", "Email": "aman@example.com", "Region": "West"},
]

## 3. Build the DataFrame and inspect it

### Why do this first?

Before cleaning, we need a baseline. Knowing the number of rows, the available columns, and their data types helps us spot problems and later confirm what changed.

### Functions and arguments

- `pd.DataFrame(raw_records)` turns a list of dictionaries into a table. Each dictionary becomes one row.
- `head(n=5)` shows the first five rows by default. Use `head(10)` to show ten.
- `shape` returns `(rows, columns)`.
- `dtypes` shows the type stored in each column.
- `describe()` summarises numerical columns. `include="all"` also includes non-numeric columns, where appropriate.

### Live coding

Create `orders_raw` from `raw_records`. Display the first five rows, shape, data types, and a summary of all columns.

In [5]:
# Create orders_raw from raw_records
orders_raw=pd.DataFrame(raw_records)

# Display the first five rows
print(orders_raw.head())

# Print the number of rows and columns
print(f"Rows: {orders_raw.shape[0]}")
print(f"Columns: {orders_raw.shape[1]}")

# Display column data types
print(orders_raw.dtypes)

# Summarise numerical and non-numerical columns
print(orders_raw.describe(include="all"))

   Order_ID  Order_Date Customer_ID            Product Quantity Unit_Price  \
0  ORD-1001  2026-01-02        C101    wireless mouse         2      15.75   
1  ORD-1002  2026-01-02        C102           Laptop X        1     999.50   
2  ORD-1003  2026-01-03        C103           Keyboard        2      45.00   
3  ORD-1004  2026-01-03        C104          Monitor Y        1     189.99   
4  ORD-1005  2026-01-04        C105          USB-C Hub        3      29.99   

      Status                  Email Region  
0    shipped  riya.sharma@gmail.com  North  
1    Pending       AMAN@EXAMPLE.COM  north  
2   SHIPPED          neha@gmail.com  South  
3  delivered                     .    West  
4    Pending          kabir@shop.in   East  
Rows: 51
Columns: 9
Order_ID       object
Order_Date     object
Customer_ID    object
Product        object
Quantity       object
Unit_Price     object
Status         object
Email          object
Region         object
dtype: object
        Order_ID  Order_Date 

## 4. Keep an untouched copy

### Why?

Cleaning often involves changing values or removing rows. Keeping a raw copy lets us compare results, repeat the process, and return to the original if a cleaning decision needs to be changed.

### Function: `copy()`

`DataFrame.copy()` creates a separate DataFrame. We will use `orders` as our working version and keep `orders_raw` unchanged.

### Live coding

Create `orders` as a copy of `orders_raw`, then display its shape.

In [6]:
# Make a working copy of orders_raw
orders= orders_raw.copy()

# Check the shape of the copy
print(orders.shape)

(51, 9)


# Part B – Detecting Missing Values

## 5. Standardise custom missing-value markers

### Why?

Pandas recognises `NaN` and `None` as missing, but it cannot automatically know that `"."`, `"NA"` or `"-999"` are placeholders in a particular dataset. If we leave these codes in place, they may be treated as real text or numbers.

### Functions and arguments

- `replace(to_replace, value)` substitutes specified values. For example, `replace([".", "NA"], np.nan)` replaces either text marker with a missing value.
- `str.strip()` removes spaces at the beginning and end of text.
- `replace(r"^\s*$", np.nan, regex=True)` replaces strings containing only whitespace. `regex=True` tells pandas to interpret the pattern as a regular expression.
- `np.nan` represents a missing numeric value and is also commonly used in pandas columns.

### Live coding

1. Replace `"."` and `"NA"` in `Email` with `np.nan`.
2. Replace `"-999"` in `Unit_Price` with `np.nan`.
3. Strip spaces from `Email`.
4. Replace blank email strings with `np.nan`.

In [7]:
# Replace the custom missing markers in Email and Unit_Price
custom_markers=[".", "NA", "-999"]
orders["Email"] = orders["Email"].replace(custom_markers, np.nan)
orders["Unit_Price"] = orders["Unit_Price"].replace(custom_markers, np.nan)

# Remove leading and trailing spaces from Email
orders["Email"] = orders["Email"].str.strip()

# Convert empty or whitespace-only email values to np.nan
orders["Email"]=orders["Email"].replace(r"^\s*$", np.nan, regex=True)
orders.loc[orders["Email"].isna(), "Email"]

,Email
5,NaN
8,NaN
17,NaN
25,NaN
31,NaN
36,NaN
40,NaN
45,NaN


## 6. Count missing values and calculate their percentage

### Why?

A count tells us how many values are missing. A percentage helps compare columns of different sizes. This is useful when deciding whether a field can be repaired, should be removed, or needs further investigation.

### Functions and arguments

- `isna()` and `isnull()` return `True` where values are missing. They are equivalent.
- `sum()` counts `True` values when applied to the Boolean result.
- `mean()` on a Boolean mask gives the proportion of missing values because `True` is treated as 1 and `False` as 0.
- `sort_values(ascending=False)` puts the largest values first.

### Live coding

Display missing-value counts and percentages for every column. Sort the percentage report from highest to lowest.

In [8]:
# Count missing values in each column
missing_counts = orders.isna().sum()
print(missing_counts)

# Calculate missing percentages and sort them in descending order
missing_percentage=(orders.isna().mean()*100).sort_values(ascending=False)
print(missing_percentage)

Order_ID       0
Order_Date     0
Customer_ID    0
Product        0
Quantity       0
Unit_Price     1
Status         0
Email          8
Region         0
dtype: int64
Email          15.686275
Unit_Price      1.960784
Order_ID        0.000000
Order_Date      0.000000
Customer_ID     0.000000
Quantity        0.000000
Product         0.000000
Status          0.000000
Region          0.000000
dtype: float64


## 7. Inspect rows with missing values

### Why?

A missing-value count tells us where the problem is, but looking at actual records can help us decide what to do. For example, an email may be optional, while a price may be necessary for a sales calculation.

### Functions and arguments

- `isna()` creates a Boolean mask.
- `|` means OR, so `Email.isna() | Unit_Price.isna()` selects rows where either field is missing.
- `loc[condition, columns]` selects rows matching a condition and only the requested columns.

### Live coding

Display records where `Email` or `Unit_Price` is missing. Show only `Order_ID`, `Product`, `Unit_Price`, and `Email`.

In [9]:
# Select rows where Email OR Unit_Price is missing
missing_rows=orders[orders["Email"].isna() | orders["Unit_Price"].isna()]

# Display only the four requested columns
print(missing_rows[["Order_ID", "Product", "Unit_Price", "Email"]])


    Order_ID         Product Unit_Price            Email
5   ORD-1006        Laptop X     999.50              NaN
8   ORD-1009        Keyboard      45.00              NaN
10  ORD-1011        Laptop X        NaN  arjun@gmail.com
17  ORD-1018       Monitor Y     189.99              NaN
25  ORD-1025        Keyboard      45.00              NaN
31  ORD-1031       Monitor Y     189.99              NaN
36  ORD-1036       Desk Lamp      22.50              NaN
40  ORD-1040  Wireless Mouse      15.75              NaN
45  ORD-1045        Laptop X     999.50              NaN


# Part C – Dropping and Filling Missing Values

## 8. Remove rows or columns with `dropna()`

### Why?

Sometimes a record cannot be used for a particular task because an essential field is missing. In other cases, one missing value is not enough reason to discard an otherwise useful record. `dropna()` gives us control over what is removed.

### Important arguments

- `axis=0` works on rows (the default); `axis=1` works on columns.
- `how="any"` removes a row or column if at least one value is missing.
- `how="all"` removes it only if all values are missing.
- `thresh=N` keeps a row or column only if it contains at least `N` non-missing values.
- `subset=["Email"]` checks missingness only in the specified column(s).

### Live coding

1. Create `orders_complete_email` by removing rows where `Email` is missing.
2. Create `orders_not_empty` by removing rows where every field is missing.
3. Compare the shapes of the DataFrames.

In [10]:
# Drop rows where Email is missing, storing the result in orders_complete_email
orders_complete_email=orders.dropna(subset=["Email"])

# Remove rows where all fields are missing
orders_cleaned=orders.dropna(how="all")

# Compare the shapes of the original and resulting DataFrames
print(orders.shape)
print(orders_complete_email.shape)
print(orders_cleaned.shape)

(51, 9)
(43, 9)
(51, 9)


## 9. Fill missing prices with the median

### Why?

A missing price should not automatically be replaced with zero: zero would mean the product was free. The median provides a simple numeric replacement that is less sensitive to very high prices than the mean.

In a real business workflow, first check whether the correct price can be recovered from a product catalogue. Median imputation is a fallback for practice, not always the right business rule.

### Functions and arguments

- `median()` returns the middle value of a numeric column after ordering the values.
- `fillna(value)` fills missing entries with the supplied value.
- `isna().sum()` checks how many missing values remain.

### Live coding

Find the median `Unit_Price`, fill missing prices with it, and check the remaining missing-value count.

In [13]:
# Convert Unit_Price to numeric first
orders["Unit_Price"] = pd.to_numeric(orders["Unit_Price"], errors="coerce")

# Calculate the median Unit_Price
median_price = orders["Unit_Price"].median()

# Fill missing Unit_Price values with the median
orders["Unit_Price"] = orders["Unit_Price"].fillna(median_price)

# Count missing Unit_Price values after filling
print(orders["Unit_Price"].isna().sum())

0


## 10. Fill a missing category using the mode

### Why?

Mean and median are not meaningful for labels such as region or status. The mode is the most frequent value and can be used as a simple replacement when that choice makes sense.

### Functions and arguments

- `value_counts()` counts occurrences of each category.
- `mode()` returns the most frequent value or values.
- `mode().iloc[0]` selects the first mode if more than one value is tied.
- `fillna()` fills missing entries.

### Live coding

Find the most frequent `Region`, fill missing region values with it, and display the region counts.

In [15]:
# Find the most frequent Region
mode_region = orders["Region"].mode().iloc[0]

# Fill missing Region values using the mode
orders["Region"] = orders["Region"].fillna(mode_region)

# Display the region counts
print(orders["Region"].value_counts())

Region
North    13
South    13
East     13
West     11
north     1
Name: count, dtype: int64


## 11. Use a group-specific median

### Why?

Products have very different prices. If a laptop price is missing, replacing it with the median across laptops, keyboards, and lamps may be misleading. A group-specific median uses the typical price for the same product.

### Functions and arguments

- `groupby("Product")` separates rows by product.
- `["Unit_Price"]` selects the price column from each group.
- `transform("median")` calculates a median for each group and returns values aligned to the original rows.
- `fillna(...)` uses those aligned medians to fill missing prices.

### Live coding

Create a copy named `orders_group_fill`. Fill missing prices using the median of the same product, then check whether any prices remain missing.

In [17]:
# Create a separate copy for group-based imputation
orders_group_fill=orders.copy()

# Fill missing prices with the median for their Product group
orders_group_fill["Unit_Price"] = orders_group_fill.groupby("Product")["Unit_Price"].transform("median")

# Check the number of missing prices remaining
print(orders_group_fill["Unit_Price"].isna().sum())

0


## 12. Forward fill, backward fill, and interpolation

### Why?

For ordered measurements such as daily stock levels or sensor readings, missing values may be estimated from neighbouring observations. These techniques should not be applied blindly to unrelated rows, because row order must have a meaningful sequence.

### Methods

- `ffill()` carries the previous valid value forward.
- `bfill()` carries the next valid value backward.
- `interpolate(method="linear")` estimates missing numeric values between known values using a straight-line change.

### Live coding

Create a Series from `[120, np.nan, 150, np.nan, 180]`. Display the original Series and the results of forward fill, backward fill, and linear interpolation.

In [23]:
# Create the example Series
s = pd.Series([120, np.nan, 150, np.nan, 180])
print(s)

# Apply forward fill
print( s.ffill())

# Apply backward fill
print( s.bfill())

# Apply linear interpolation
print(s.interpolate(method="linear"))

0    120.0
1      NaN
2    150.0
3      NaN
4    180.0
dtype: float64
0    120.0
1    120.0
2    150.0
3    150.0
4    180.0
dtype: float64
0    120.0
1    150.0
2    150.0
3    180.0
4    180.0
dtype: float64
0    120.0
1    135.0
2    150.0
3    165.0
4    180.0
dtype: float64


# Part D – Duplicates, Replacement, and Column Names

## 13. Detect and remove exact duplicates

### Why?

Duplicate records may be caused by repeated exports, repeated form submissions, or accidentally appending the same file twice. If they are counted twice, totals and reports may be incorrect.

### Functions and arguments

- `duplicated()` returns `True` for rows that repeat an earlier row.
- `duplicated().sum()` counts duplicate rows.
- `drop_duplicates()` removes repeated rows and keeps the first occurrence by default.
- `keep="last"` keeps the last occurrence instead.
- `subset=[...]` checks duplicates using only selected columns.

### Live coding

Count exact duplicate rows, create `orders_dedup` without exact duplicates, and compare row counts.

In [ ]:
# Count exact duplicate rows


# Remove exact duplicate rows into orders_dedup


# Compare row counts before and after

## 14. Remove duplicates using a business key

### Why?

Two rows do not need to be identical across every column to represent the same business record. Sometimes a key such as an order-line ID identifies the record.

**Be careful:** a customer can place many orders, so `Customer_ID` is not a valid unique key for an order table. In this example, we will demonstrate duplicate handling using `Order_ID`, but only do this when the data design confirms that one order ID should identify one record.

### Live coding

Display rows with repeated `Order_ID` values. Then create `orders_by_id` by keeping the last row for each ID and check for remaining duplicate IDs.

In [ ]:
# Display rows whose Order_ID is duplicated


# Drop duplicate Order_ID values while keeping the last occurrence


# Check for any duplicate Order_ID values remaining

## 15. Understand `replace()` versus `fillna()` and use mappings

### Why?

`fillna()` is designed for missing values. `replace()` can also substitute invalid codes or inconsistent labels. `map()` is useful when known values should be translated using a dictionary.

### Examples

- `replace(-999, np.nan)` changes a known invalid numeric code into a missing value.
- `replace({"shipped": "Shipped"})` replaces one specific label.
- `map({"N": "North", "S": "South"})` translates values using a mapping dictionary. Values absent from the dictionary become missing, so check that the mapping covers the categories you need.

### Live coding

Create a separate copy. Replace any remaining `-999` price with `np.nan`, standardise `Status` labels, and make `"north"` and `"North"` consistent in `Region`.

In [ ]:
# Create a copy for replacement practice


# Replace any remaining invalid Unit_Price code


# Standardise Status labels


# Standardise Region labels

## 16. Rename columns

### Why?

Consistent column names make code easier to read and reduce typing mistakes. In shared projects, teams often use lowercase names and underscores instead of spaces.

### Function and arguments

`rename(columns={"old_name": "new_name"})` changes selected column names using a dictionary. To apply a function to every column name, `rename(columns=str.lower)` converts all labels to lowercase.

### Live coding

Create `orders_named` with lowercase column names. Then display the new column names. If any names contain spaces, replace those spaces with underscores.

In [ ]:
# Rename all column names to lowercase


# Replace spaces in column names with underscores if needed


# Display the new column names

# Part E – Outlier Checks

## 17. Identify possible outliers using the IQR rule

### Why?

An outlier is a value that is unusually far from most observations. It may be a data-entry error, a special case, or a genuine high-value order. We flag outliers for review rather than automatically deleting them.

### Terms and calculations

- `quantile(0.25)` returns the first quartile, Q1.
- `quantile(0.75)` returns the third quartile, Q3.
- `IQR = Q3 - Q1`.
- Lower boundary: `Q1 - 1.5 * IQR`.
- Upper boundary: `Q3 + 1.5 * IQR`.

### Live coding

Calculate Q1, Q3, and IQR for `Unit_Price`. Find the lower and upper boundaries, then display rows outside these boundaries.

In [ ]:
# Calculate Q1 and Q3


# Calculate the IQR


# Calculate lower and upper boundaries


# Display possible outlier rows

## 18. Use a simple z-score check

### Why?

The z-score tells us how far a value is from the mean in units of standard deviation. An absolute z-score greater than 3 is a common quick flag, although this method can be misleading when the data is heavily skewed.

### Functions

- `mean()` calculates the average.
- `std()` calculates the standard deviation.
- `abs()` returns absolute values.
- Boolean filtering selects the rows meeting a condition.

### Live coding

Calculate z-scores for `Unit_Price` and display rows with an absolute z-score greater than 3. Exclude missing values from the calculation.

In [ ]:
# Calculate the z-score for Unit_Price


# Select rows where the absolute z-score is greater than 3

## 19. Cap extreme values for comparison

### Why?

Instead of removing an entire record, an analyst may cap a numeric value at a chosen boundary. This is sometimes called winsorisation-style treatment. It should only be used when justified by the task and should not replace investigation of data-entry errors.

### Function

`clip(lower=..., upper=...)` limits values to a lower and upper boundary. Values below the lower boundary are set to the lower boundary; values above the upper boundary are set to the upper boundary.

### Live coding

Create `Unit_Price_Capped` using the IQR boundaries from the previous section. Compare the original and capped values for the first ten rows.

In [ ]:
# Cap Unit_Price using the lower and upper IQR boundaries


# Compare original and capped prices

# Part F – Binning, Indicator Columns, and Sampling

## 20. Create price categories with `pd.cut()`

### Why?

A report may be easier to interpret if prices are grouped into categories such as Budget or Premium. `cut()` divides a numeric range using specified boundaries. The intervals are based on the boundaries you provide, not on how frequently values occur.

### Arguments

- `x`: the numeric Series to divide.
- `bins`: the boundary values.
- `labels`: the category labels for the intervals.
- `include_lowest=True`: includes the lowest boundary in the first interval.

### Live coding

Create `Price_Band` with Budget (up to 25), Standard (above 25 to 100), Premium (above 100 to 500), and High-value (above 500). Display the product, price and category.

In [ ]:
# Define the price boundaries and category labels


# Use pd.cut() to create Price_Band


# Display Product, Unit_Price, and Price_Band

## 21. Create equal-frequency categories with `pd.qcut()`

### Why?

When a numeric column is strongly skewed, fixed-width bins may leave some groups almost empty. `qcut()` uses quantiles to create groups with roughly similar numbers of observations.

### Arguments

- `q=4` requests four quantile groups.
- `labels=[...]` supplies labels for the groups.
- `duplicates="drop"` allows duplicate boundaries to be dropped if many values are identical. If boundaries are dropped, the number of labels must match the number of resulting bins.

### Live coding

Divide `Unit_Price` into four quantile groups labelled `Q1` to `Q4`. Display the count in each group. If duplicate boundaries cause an error, inspect the data and adjust the bins or labels.

In [ ]:
# Create four quantile-based price groups using pd.qcut()


# Display the number of records in each group

## 22. Create indicator columns with `pd.get_dummies()`

### Why?

Many machine-learning algorithms work with numbers rather than text labels. One-hot encoding creates a separate 0/1 column for each category. This avoids making categories such as `"Pending"` and `"Shipped"` look like ordered numbers.

### Arguments

- `data`: the Series or DataFrame to encode.
- `prefix="status"` adds a prefix to the new column names.
- `dtype=int` makes the indicator values integers (0 or 1).

### Live coding

Create one-hot encoded columns for `Status` with a prefix. Then create another indicator DataFrame for `Price_Band`.

In [ ]:
# One-hot encode Status with a column-name prefix


# Display the first five encoded rows


# One-hot encode Price_Band

## 23. Shuffle and sample rows

### Why?

Sampling lets us inspect a smaller part of a large dataset. Shuffling randomises row order and can be useful before model training or splitting data into sets.

### Functions and arguments

- `np.random.permutation(len(df))` generates a random ordering of row positions.
- `sample(n=5)` selects five rows.
- `sample(frac=0.2)` selects 20% of the rows.
- `random_state=42` makes the sample reproducible.
- `replace=True` allows a row to be selected more than once.

### Live coding

Shuffle the row order, select five rows with a fixed random state, select 20% of the rows, and try a sample with replacement.

In [ ]:
# Shuffle the rows using np.random.permutation


# Select five rows reproducibly


# Select 20% of the rows reproducibly


# Select rows with replacement

# Part G – String Cleaning and Pattern Matching

## 24. Basic Python string operations

### Why?

A single text value may contain extra spaces, separators, or several pieces of information. Python's built-in string methods can clean and split one string before we apply similar operations to an entire column.

### Methods

- `strip()` removes whitespace at the beginning and end.
- `split(",")` splits a string at commas and returns a list.
- `" | ".join(parts)` combines a list of strings with a separator.
- `find(",")` returns the position of the first comma, or `-1` if it is not found.
- `replace(",", "::")` replaces commas with another string.

### Live coding

Use the sample text to try each of these operations and print the results.

In [ ]:
example_text = "  ORD-1051, Laptop X, Pending  "

# Remove leading and trailing spaces


# Split the text at commas


# Join the parts using " | "


# Find the first comma


# Replace commas with "::"

## 25. Clean an entire text column with `.str`

### Why?

Applying ordinary Python string methods one row at a time is unnecessary for a pandas Series. The `.str` accessor applies vectorised string operations across a column and generally handles missing values more conveniently.

### Useful methods

- `.str.strip()` removes leading and trailing spaces.
- `.str.lower()` converts text to lowercase.
- `.str.upper()` converts text to uppercase.
- `.str.title()` capitalises the first letter of each word.

Use a consistent rule for each field. For example, product and status labels can use title case, while email addresses are commonly normalised to lowercase.

### Live coding

Create cleaned versions of `Product`, `Status`, and `Email`. Display the original and cleaned values side by side for the first few records.

In [ ]:
# Clean Product by removing spaces and standardising case


# Clean Status using strip() and title()


# Clean Email using strip() and lower()


# Display original and cleaned values side by side

## 26. Find values that match a pattern with `.str.contains()`

### Why?

Pattern matching is useful when we want to find product names containing a word, email addresses from a provider, or records containing a particular code.

### Arguments

- `.str.contains(pattern)` checks whether each string contains a pattern.
- `case=False` makes the match case-insensitive.
- `na=False` treats missing values as non-matches instead of returning missing Boolean results.
- `regex=False` treats the pattern as ordinary text rather than a regular expression.

### Live coding

Find products containing `"laptop"` and email addresses containing `"gmail"`. Use `case=False` and `na=False`.

In [ ]:
# Filter laptop products using str.contains()


# Filter Gmail addresses using str.contains()

## 27. Use regular expressions with `re`

### Why?

Regular expressions describe patterns in text. They are useful when text has a recognisable structure but the exact values vary, such as email addresses, IDs, or inconsistent whitespace.

### Functions

- `re.findall(pattern, text)` returns all matches.
- `re.search(pattern, text)` finds the first match anywhere in the text.
- `re.match(pattern, text)` checks for a match at the beginning.
- `re.split(pattern, text)` splits text wherever the pattern matches.
- `re.sub(pattern, replacement, text)` replaces every matching part.
- `re.IGNORECASE` makes matching case-insensitive.

A regular expression can be powerful, but keep it as simple as the task allows. A basic email pattern is useful for classroom practice, but it is not a complete email validation standard.

### Live coding

Extract email-like strings from the sample text, replace repeated whitespace with a single space, and split a string on commas or semicolons.

In [ ]:
sample_text = "Contact riya@example.com or help@shop.in for order ORD-1051."

# Write a simple pattern for email-like text


# Extract all matches using re.findall()


# Replace repeated whitespace using re.sub()


# Split a string on commas or semicolons using re.split()

## 28. Extract fields from a string with `.str.extract()`

### Why?

An identifier such as `ORD-1051` contains both a prefix and a numeric part. Extracting these into separate columns makes it easier to group, filter, or validate records.

### How capture groups work

In a regular expression, parentheses `(...)` define capture groups. Each group becomes a separate output column when used with `Series.str.extract()`.

For example, a pattern can capture letters before a hyphen and digits after it.

### Live coding

Create a Series with IDs such as `ORD-1051`, `ORD-1052`, and `INV-2050`. Use `.str.extract()` to split each ID into its prefix and numeric portion.

In [ ]:
# Create a Series of sample IDs


# Extract the alphabetic prefix and numeric part with str.extract()


# Display the extracted columns

## 29. Split email addresses into user and domain

### Why?

Sometimes the original field contains multiple useful pieces of information. Separating an email address into its username and domain can support simple quality checks or summaries.

### Methods

- `.str.split("@")` splits each email at the `@` symbol.
- `.str.get(0)` selects the first part.
- `.str.get(1)` selects the second part.

This assumes the address has the expected structure. Invalid or missing addresses should be checked rather than silently treated as valid.

### Live coding

Create `Email_User` and `Email_Domain` from the `Email` column. Display the original email with the two new columns.

In [ ]:
# Extract the username before @ into Email_User


# Extract the domain after @ into Email_Domain


# Display Email, Email_User, and Email_Domain

# Part H – Build a Repeatable Cleaning Pipeline

## 30. Put the cleaning steps together

### Why?

During practice, we apply individual methods to understand them. In an actual project, these steps need to work together in a predictable order. A pipeline also makes it easier to repeat the same rules when a new batch of data arrives.

A sensible sequence is:

1. Start from a fresh copy of the raw data.
2. Standardise custom missing-value markers.
3. Clean text fields.
4. Convert numeric and date columns.
5. Remove exact duplicate records.
6. Handle missing values using a reasoned rule.
7. Create derived columns.
8. Run quality checks.
9. Save the prepared data.

### Live coding

Create `orders_clean` from `orders_raw` and implement the steps below. For missing prices, use the median of the same product first, then the overall median only if values remain missing. This is a practice rule; a real business may prefer to retrieve the correct price from a product catalogue.

In [ ]:
# Start from a fresh copy of the raw data
orders_clean = orders_raw.copy()

# 1. Replace custom missing markers in Email and Unit_Price


# 2. Strip spaces and standardise text columns


# 3. Convert Quantity and Unit_Price to numeric with errors="coerce"


# 4. Convert Order_Date to datetime with errors="coerce"


# 5. Remove exact duplicate rows


# 6. Fill missing prices using product medians, then the overall median if needed


# 7. Create Line_Total = Quantity * Unit_Price


# 8. Create High_Value_Order for line totals above 500


# 9. Check missing values, duplicate rows, and data types


# 10. Save as orders_cleaned.csv without the DataFrame index

# Part I – Guided Practice

Complete each question in the code cell immediately below it. The questions use the same order dataset, but each one practises a slightly different decision or function. Add short comments explaining what your code is doing.

## Question 1 – Create a missing-value report

### Why?

A compact report makes it easier to prioritise fields that need attention. A percentage is more useful than a count alone when comparing columns.

### Your task

Create `quality_report` with one row per column and these fields:

- Data type
- Non-missing count
- Missing count
- Missing percentage

Sort the report by missing percentage from highest to lowest.

### Functions to consider

`dtypes`, `count()`, `isna().sum()`, `isna().mean()`, `DataFrame`, and `sort_values(ascending=False)`.

In [ ]:
# Write your solution for Question 1 below.

## Question 2 – Keep rows with enough information

### Why?

Sometimes a row has one missing field but is still useful. `thresh` lets us retain rows that have a minimum number of non-missing values.

### Your task

Create `orders_mostly_complete` by keeping rows with at least 7 non-missing values. Compare its shape with `orders_raw`.

### Functions to consider

`dropna(thresh=7)` and `shape`.

In [ ]:
# Write your solution for Question 2 below.

## Question 3 – Check duplicates by customer

### Why?

A customer can place many different orders. This means that repeated `Customer_ID` values are not automatically duplicate orders.

### Your task

Count the number of rows associated with each customer. Identify any customers with more than one row. In a comment, explain why removing duplicates based only on `Customer_ID` could delete valid data.

### Functions to consider

`value_counts()` or `groupby()`, `size()`, and Boolean filtering.

In [ ]:
# Write your solution for Question 3 below.

## Question 4 – Find invalid prices

### Why?

Special codes can remain in a numeric column if they were not converted during import. Negative prices may also be invalid, depending on the business rules.

### Your task

Create a copy of `orders_raw`, replace `-999` in `Unit_Price` with `np.nan`, convert the column to numeric, and display rows where the price is missing or less than or equal to zero.

### Functions to consider

`copy()`, `replace()`, `pd.to_numeric(errors="coerce")`, `isna()`, and Boolean filtering.

In [ ]:
# Write your solution for Question 4 below.

## Question 5 – Summarise sales by product

### Why?

After the data has been cleaned, grouping can help answer business questions such as which products sold the most units and which generated the most revenue.

### Your task

Using `orders_clean` if you have completed the pipeline, group by `Product` and calculate total quantity, average unit price, and total sales. Sort products by total sales from highest to lowest.

### Functions to consider

`groupby()`, `agg()`, `sum()`, `mean()`, and `sort_values()`.

In [ ]:
# Write your solution for Question 5 below.

## Question 6 – Flag suspicious email values

### Why?

A non-missing email is not necessarily a valid-looking email. Simple pattern checks can identify records for review without deleting them automatically.

### Your task

Create a cleaned email Series by stripping spaces and converting text to lowercase. Flag rows where an email is present but does not contain `"@"`. Display the flagged rows.

### Functions to consider

`.str.strip()`, `.str.lower()`, `.str.contains("@", na=False)`, and Boolean filtering.

In [ ]:
# Write your solution for Question 6 below.

## Question 7 – Create an indicator matrix

### Why?

Categorical labels are easy for people to read, but many models need numeric inputs. One-hot encoding represents each category using a separate indicator column.

### Your task

Create one-hot encoded columns for `Region` and `Status`. Display the first five rows and the new column names.

### Functions to consider

`pd.get_dummies()`, `prefix=`, and `dtype=int`.

In [ ]:
# Write your solution for Question 7 below.

## Question 8 – Review high-value order lines

### Why?

A high line total may represent a genuine laptop sale rather than a data error. Outlier checks help us find unusual records that deserve inspection.

### Your task

Create `Line_Total` from quantity multiplied by unit price in a copy of the cleaned dataset. Apply the IQR rule to `Line_Total`, display the potential outliers, and write a comment explaining why you would inspect rather than automatically delete them.

### Functions to consider

`quantile()`, `IQR = Q3 - Q1`, Boolean filtering, and `copy()`.

In [ ]:
# Write your solution for Question 8 below.

## Question 9 – Take a reproducible sample

### Why?

A sample helps us inspect a large dataset quickly. Setting `random_state` makes the same sample reproducible, which is helpful when demonstrating or debugging code with others.

### Your task

Select 10% of the rows using `sample(frac=0.1, random_state=42)`. Run the same command twice and compare the results.

### Functions to consider

`sample()`, `frac=`, and `random_state=`.

In [ ]:
# Write your solution for Question 9 below.

## Question 10 – Prepare a final quality checklist

### Why?

A cleaning process should finish with evidence that the output is in a usable state. A few simple checks can catch issues before the dataset is shared with another analyst.

### Your task

For `orders_clean`, report:

1. Number of rows and columns
2. Missing values per column
3. Number of exact duplicate rows
4. Data types
5. Whether `Line_Total` contains any missing values

Write a short comment stating what you would investigate if missing values or duplicates remain.

### Functions to consider

`shape`, `isna().sum()`, `duplicated().sum()`, `dtypes`, and `isna().any()`.

In [ ]:
# Write your solution for Question 10 below.

# Quick Revision

Use this section to check whether you can explain not only how a function works, but also why you would choose it.

| Task | Function or method | Useful arguments / notes |
|---|---|---|
| Read or create tabular data | `pd.read_csv()`, `pd.DataFrame()` | `na_values` can identify custom missing markers during CSV import |
| Find missing values | `isna()`, `isnull()`, `notna()`, `notnull()` | Use `sum()` for counts and `mean() * 100` for percentages |
| Remove missing data | `dropna()` | `axis`, `how`, `thresh`, `subset` |
| Fill missing values | `fillna()` | Constant, median, mode, or values calculated by group |
| Replace special codes | `replace()` | Handles invalid values and arbitrary substitutions, not only missing values |
| Find duplicates | `duplicated()` | `subset` checks selected key columns |
| Remove duplicates | `drop_duplicates()` | `subset`, `keep="first"` or `keep="last"` |
| Rename columns | `rename()` | `columns={...}` or `columns=str.lower` |
| Convert data types | `pd.to_numeric()`, `pd.to_datetime()` | `errors="coerce"` converts invalid values to missing values |
| Group records | `groupby()`, `agg()`, `transform()` | `transform()` returns values aligned with the original rows |
| Find outliers | `quantile()`, `mean()`, `std()`, `abs()` | IQR bounds and z-scores flag unusual values |
| Limit extreme values | `clip()` | `lower` and `upper` set the boundaries |
| Create categories | `pd.cut()`, `pd.qcut()` | Fixed intervals versus quantile-based intervals |
| Encode categories | `pd.get_dummies()` | `prefix`, `dtype` |
| Shuffle or sample | `np.random.permutation()`, `sample()` | `n`, `frac`, `replace`, `random_state` |
| Clean text | `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.title()` | Vectorised operations across a Series |
| Search text | `.str.contains()` | `case`, `na`, `regex` |
| Extract text | `.str.extract()`, `.str.split()`, `.str.get()` | Capture groups, delimiters, and selected split positions |
| Work with regex | `re.findall()`, `re.search()`, `re.match()`, `re.split()`, `re.sub()` | Pattern-based search, splitting, and replacement |
| Save results | `to_csv()` | `index=False` avoids saving the DataFrame index as a separate column |

## Final reminder

Cleaning is not just about making errors disappear. It is about making data consistent and fit for a specific purpose. Keep the raw data, understand what each column means, choose a method that suits the problem, and verify the result before using it for analysis.